## **Data Quality**

In [ ]:
import spacy
from collections import Counter

# Load spacy model 
nlp = spacy.load("en_core_web_sm")

def analyze_text_quality(text):
    doc = nlp(text)

    # Check for spelling errors
    misspelled = [
        token.text for token in doc if token._.is_misspelled
    ]

    # Check for grammatical issues
    pos_counts = Counter(token.ppos_ for token in doc)
    grammar_score = pos_counts['NOUN'] + pos_counts['VERB'] + pos_counts['ADJ'] + pos_counts['ADV']

    # Check for sentence completence
    incomplete_sentences = [
        sent.text for sent in doc.sents if len(sent) < 3
    ]
    return {
        "misspelled_words":misspelled,
        "grammar_score":grammar_score,
        "incomplete_sentence":incomplete_sentences,
            }

In [ ]:
# Example usage of code
text = "This is a sample txt with sum issues. incomplet"
quality_report = analyze_text_quality(text)
print(quality_report)

## **Data Preprocessing**

In [ ]:
import unicodedata
import re
from nltk.tokenize import word_tokkenize
from nltk.corpus import stopwords
import nltk

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Define the preprocessing Function
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()

    # Normalize unicode character
    text = unicodedata.normalize(
        'NFKD', text
    ).encode('utf-8')

    # Remove punctunation
    text = re.sub(r'[^\w\s]', '', text)

    # Normalize whitespace
    text = ' '.join(text.split())

    # Tokenize
    tokens = word_tokkenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [
        token for token in tokens if token not in stop_words
    ]
    preprocessed_text = ' '.join(tokens)
    return preprocessed_text

In [ ]:
# Example of usage code
raw_text = "This is an EXAMPLE of text preprocessing... It's quite useful!"
clean_text = preprocess_text(raw_text)
print(clean_text)

## **Multilingual and  Code-mixed Data**

In [ ]:
from langdetect import detect
from unidecode import unidecode
from nltk import word_tokenizer
import nltk

# Download required NLTK data
nltk.download('punkt')

def handle_multilingual_text(text):
    # Detect language 
    try:
        lang = detect(text)
    except:
        lang = 'unknown'

    # Transliterate non_ASCII characters
    transliterated_text = unidecode(text)

    # Tokenize using NLTK
    tokens = word_tokkenize(transliterated_text)
    return {
        'original': text,
        'language': lang,
        'transliterated': transliterated_text,
        'tokens': tokens
    }

## **Deduplication Techniques**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def deduplicate_corpus(corpus, similarity_threshold=0.9):
    # Create TF_IDF vectorizer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    # Compute pairwise similarities
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Find duplicates
    duplicates = set()
    for i in range(len(corpus)):
        for j in range(i + 1 , len(corpus)):
            if similarity_matrix[i, j] > similarity_threshold:
                duplicates.add(j)

    # Create a deduplication corpus
    deduplicated_corpus = [
        doc for i, doc in enumerate(corpus)
        if i not in duplicates
    ]
    return deduplicate_corpus

## **Automated Data Cleaning**

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download required NLTK data
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

class DataCleaningPipeline:
    def  __init__(self, similarity_threshold=0.9, min_length=10, max_length=1000):
        self.similarity_threshold = similarity_threshold
        self.min_length = min_length
        self.max_length = max_length
        self.vectorizer = TfidfVectorizer(stop_words='english')


    def preprocess(self, text):
        # Basic preprocess
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        tokens = [
            word for word in text.split()
            if word not in stop_words
        ]
        return ' '.join(tokens)
    

    def filter_by_length(self, df):
        return df[
            (df['text'].str.len() >= self.min_length) &
            (df['text'].str.len() <= self.max_length)
        ]


    def deduplicate(self, df):
        tfodf_matrix = self.vectorizer.fit_transform(df['text'])
        similarity_matrix = cosine_similarity(tfodf_matrix)
        duplicates = set()
        for i in range(len(df)):
            for j in range(i + 1, len(df)):
                if similarity_matrix[i, j] > self.similarity_threshold:
                    duplicates.add(j)
        return df.drop(df.index[list(duplicates)])
    

    def clean(self, input_file, output_file):
        # Read Data
        df = pd.read_csv(input_file)
        # Preprocess
        df['text'] = df['text'].apply(self.preprocess)
        # Filter by length
        df = self.filter_by_length(df)
        # Deduplicate
        df = self.deduplicate(df)
        # Save Clean Data
        df.to_csv(output_file, index=False)
        print(f"Clean data saved to {output_file}")

In [ ]:
# For use
pipeline = DataCleaningPipeline()
pipeline.clean('input_data.csv', 'output_data.csv')

## **Data Validation**

In [ ]:
def validate_cleaned_data(file_path, sample_size=100):
    df = pd.read_csv(file_path)
    # Basic statistics
    print(f"Total samples : {len(df)}")
    print(f"Average text length : "
          f"{df['text'].str.len().mean(): .2f}")
    print(f"Unique samples : {df['text'].unique()}")



# Check for empty or very short text
short_texts = df[df['text'].str.len() < 10]    
print(f"Texts shorter than 10 characters: "
      f"{len(short_texts)}")


# Sample for a manual review
sample = df.sample(n=min(sample_size, len(df)))
print(sample['text'].head())


# Evaluate the impact on the model's perplexity
model = GPT4LMHeadModel.from_pretrained('GPT4')
tokenizer = GPT4Tokenizer.from_pretrained('GPT4')
def calculate_perplexity(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        outputs = model(inputs, labels=inputs['input_ids'])
      return torch.exp(outputs.loss).item()
sample_perplexities = sample['text'].apply(
    calculate_perplexity
)
print(f"\nAverage perplexity on sample: "
      f"{sample_perplexities.mean():.2f}")

# See an example
validate_cleaned_data('claened_data.csv')
